# MedGemma 1.5 reads a CT volume

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rekalantar/MedGemma-CT-Reader/blob/main/tutorial/medgemma_ct.ipynb)

MedGemma 1.5 is the first open multimodal LLM that takes **3D CT and MRI** rather than
single 2D images. Its vision encoder still only accepts 2D RGB, so a volume becomes a
sequence of axial slices — each one windowed into three channels and costing 256 vision
tokens, capped at 85 slices.

This notebook finds out **how much of that fits on the GPU you actually have**, then runs
the model on a real abdominal CT.

Self-contained: every function is defined here.

**Runtime → Change runtime type → GPU** before running anything.

---

**[The article](ARTICLE_URL) explains the mechanism.** This notebook measures it.

## 1 · Setup

MedGemma is a **gated** model. You need a free Hugging Face account, acceptance of the
Health AI Developer Foundations terms on the
[model page](https://huggingface.co/google/medgemma-1.5-4b-it), and a token.

In Colab: add it as a secret named `HF_TOKEN` (key icon, left sidebar) and enable it for
this notebook. Otherwise the login prompt below will ask.

In [ ]:
%%capture
!pip install -q -U transformers accelerate SimpleITK huggingface_hub

In [ ]:
import os, sys

import torch

assert torch.cuda.is_available(), "Runtime -> Change runtime type -> GPU, then rerun."

GPU = torch.cuda.get_device_name(0)
VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9
BF16 = torch.cuda.is_bf16_supported()

print(f"GPU    {GPU}")
print(f"VRAM   {VRAM:.1f} GB")
print(f"bf16   {'native' if BF16 else 'NOT supported - will use float16'}")

# Turing cards (T4) have no native bfloat16; float16 is the right choice there.
DTYPE = torch.bfloat16 if BF16 else torch.float16
print(f"dtype  {DTYPE}")

In [ ]:
# Hugging Face auth: Colab secret first, interactive login as a fallback.
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("using the HF_TOKEN Colab secret")
except Exception:
    from huggingface_hub import get_token, notebook_login
    if get_token() is None:
        notebook_login()

## 2 · A CT volume, no credentials required

MedGemma's own CT notebook pulls DICOM from Google's healthcare API, which needs a Google
Cloud login on top of the Hugging Face one. We use a NIfTI case from the MedSAM2 demo set
instead — a few MB, no auth, and the same case used in the segmentation post.

In [ ]:
import numpy as np
import SimpleITK as sitk
from huggingface_hub import hf_hub_download

DATASET = "wanglab/CT_DeepLesion-MedSAM2"
CASE = "000009_03_01_036-048"

path = hf_hub_download(DATASET, f"images/{CASE}_0000.nii.gz", repo_type="dataset")
volume_hu = sitk.GetArrayFromImage(sitk.ReadImage(path))     # (z, y, x), Hounsfield

print(f"volume {volume_hu.shape}  HU {volume_hu.min():.0f} to {volume_hu.max():.0f}")

## 3 · Three windows, three channels

A CT holds thousands of Hounsfield values; the encoder takes 256 levels per channel. The
usual approach picks one window and copies it across R, G and B — spending three channels
on one view.

MedGemma gives each channel a **different** window, so one RGB image carries three views:

| Channel | HU window | Shows |
|---|---|---|
| Red | −1024 to 1024 | morphological boundaries, bone, lung |
| Green | −135 to 215 | soft tissue |
| Blue | 0 to 80 | subtle low-contrast detail |

These are the values from MedGemma's own CT notebook.

In [ ]:
WINDOWS = ((-1024, 1024), (-135, 215), (0, 80))


def normalise(volume, low, high):
    """Clip to a Hounsfield window and rescale to 0-255."""
    out = np.clip(volume, low, high).astype(np.float32)
    out -= low
    out /= (high - low)
    return out * 255.0


def window_rgb(volume_hu, windows=WINDOWS):
    """(D, H, W) Hounsfield -> (D, H, W, 3) uint8, one window per channel."""
    stacked = np.stack([normalise(volume_hu, lo, hi) for lo, hi in windows], axis=-1)
    return np.round(stacked, 0).astype(np.uint8)


volume = window_rgb(volume_hu)
print(f"{volume_hu.shape} Hounsfield  ->  {volume.shape} uint8 RGB")

In [ ]:
# What each channel actually buys you, on real voxel values
import matplotlib.pyplot as plt

mid = len(volume) // 2
names = ["red: wide", "green: soft tissue", "blue: low contrast", "all three"]
fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
for i in range(3):
    axes[i].imshow(volume[mid][..., i], cmap="gray")
    axes[i].set_title(names[i], fontsize=11)
axes[3].imshow(volume[mid])
axes[3].set_title(names[3], fontsize=11)
for ax in axes: ax.axis("off")
plt.tight_layout()
plt.savefig("three_windows.png", dpi=150, bbox_inches="tight")

## 4 · Slices become tokens

Each slice is resized to 896×896 and encoded as **256 vision tokens**. MedGemma caps a
volume at **85 slices = 21,760 tokens**, which has to fit alongside the question and the
answer inside a 32K context. Longer volumes are sampled evenly along z rather than
truncated, so the model still sees the whole anatomy.

In [ ]:
MAX_SLICES, TOKENS_PER_SLICE = 85, 256


def sample_indices(depth, max_slices=MAX_SLICES):
    """Up to `max_slices` indices spread evenly through a volume."""
    if depth <= max_slices:
        return list(range(depth))
    return [round(i / max_slices * (depth - 1)) for i in range(1, max_slices + 1)]


keep = sample_indices(len(volume))
print(f"{len(volume)} slices -> keeping {len(keep)}"
      f"  ({len(keep) * TOKENS_PER_SLICE:,} vision tokens)")

## 5 · Build the prompt

A volume is not one input. It becomes an alternating sequence of image blocks and
`SLICE n` labels, bracketed by an instruction and a question — the labels are what let
the model point at slice positions in its answer.

In [ ]:
import base64, io
import PIL.Image


def encode_slice(slice_rgb, fmt="jpeg"):
    """One uint8 RGB slice -> a base64 data URI."""
    with io.BytesIO() as buf:
        with PIL.Image.fromarray(slice_rgb) as img:
            img.save(buf, format=fmt)
        return f"data:image/{fmt};base64," + base64.b64encode(buf.getbuffer()).decode()


INSTRUCTION = (
    "You are an expert radiologist analysing a contiguous block of axial CT "
    "slices. Review every slice provided below carefully before answering."
)


def build_messages(slices_rgb, question, instruction=INSTRUCTION):
    content = [{"type": "text", "text": instruction}]
    for n, sl in enumerate(slices_rgb, 1):
        content.append({"type": "image", "image": encode_slice(sl)})
        content.append({"type": "text", "text": f"SLICE {n}"})
    content.append({"type": "text", "text": question})
    return [{"role": "user", "content": content}]

## 6 · Load the model

In [ ]:
import transformers

MODEL_ID = "google/medgemma-1.5-4b-it"

torch.cuda.reset_peak_memory_stats()
processor = transformers.AutoProcessor.from_pretrained(MODEL_ID, use_fast=True)
model = transformers.AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=DTYPE, device_map="auto", offload_buffers=True,
)

weights_gb = torch.cuda.max_memory_allocated() / 1e9
print(f"weights on GPU: {weights_gb:.2f} GB of {VRAM:.1f} GB")
print(f"headroom:       {VRAM - weights_gb:.2f} GB for activations and KV cache")

## 7 · How many slices actually fit?

This is the question the documentation does not answer. We run the same prompt at
increasing slice counts and record peak memory and wall time, stopping at the first
out-of-memory error.

Each step is a full generate call, so this takes a few minutes.

In [ ]:
import time

QUESTION = (
    "Describe the liver in these slices. Note any hypodense lesions, giving the "
    "slice numbers where you see them. If you see none, say so."
)


def run(n_slices, max_new_tokens=256):
    """Run the model on `n_slices` evenly sampled slices. Returns a result dict."""
    idx = sample_indices(len(volume), n_slices)
    messages = build_messages(volume[idx], QUESTION)

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    start = time.time()
    try:
        inputs = processor.apply_chat_template(
            messages, add_generation_prompt=True, continue_final_message=False,
            return_tensors="pt", tokenize=True, return_dict=True,
        ).to(model.device, dtype=DTYPE)

        n_tokens = inputs["input_ids"].shape[-1]
        with torch.inference_mode():
            out = model.generate(**inputs, do_sample=False, max_new_tokens=max_new_tokens)

        text = processor.post_process_image_text_to_text(out, skip_special_tokens=True)[0]
        return dict(slices=n_slices, tokens=n_tokens, ok=True,
                    peak_gb=torch.cuda.max_memory_allocated() / 1e9,
                    seconds=time.time() - start, text=text)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        return dict(slices=n_slices, ok=False, peak_gb=float("nan"),
                    seconds=time.time() - start, tokens=None, text=None)

In [ ]:
LADDER = [4, 8, 16, 32, 48, 64, 85]
results = []

print(f"{'slices':>7}{'prompt tok':>12}{'peak GB':>10}{'secs':>8}  status")
for n in LADDER:
    r = run(n)
    results.append(r)
    status = "ok" if r["ok"] else "OUT OF MEMORY"
    tok = f"{r['tokens']:,}" if r["tokens"] else "-"
    peak = f"{r['peak_gb']:.2f}" if r["ok"] else "-"
    print(f"{n:>7}{tok:>12}{peak:>10}{r['seconds']:>8.1f}  {status}")
    if not r["ok"]:
        break

workable = [r for r in results if r["ok"]]
print(f"\nLargest slice count that fits on this {GPU}: "
      f"{workable[-1]['slices'] if workable else 0}")

In [ ]:
# Memory against slice count
ok = [r for r in results if r["ok"]]
if len(ok) > 1:
    plt.figure(figsize=(8, 3.6))
    plt.plot([r["slices"] for r in ok], [r["peak_gb"] for r in ok],
             "o-", color="#3FC1C9", linewidth=2, label="peak GPU memory")
    plt.axhline(VRAM, color="#FF6B5B", linestyle="--", label=f"{GPU} limit")
    plt.xlabel("slices sent"); plt.ylabel("peak GPU memory (GB)")
    plt.ylim(0, VRAM * 1.08); plt.legend(); plt.tight_layout()
    plt.savefig("memory_vs_slices.png", dpi=150, bbox_inches="tight")

## 8 · What it actually said

The answer from the largest slice count that fit.

In [ ]:
best = [r for r in results if r["ok"]][-1]
print(f"{best['slices']} slices, {best['tokens']:,} prompt tokens, "
      f"{best['seconds']:.0f}s\n")
print(best["text"])

## 9 · Does more context change the answer?

Worth comparing: the same question at the smallest and largest slice counts that worked.
If the answers agree, the extra slices bought nothing on this case.

In [ ]:
for r in [results[0], best]:
    if not r["ok"]:
        continue
    print(f"=== {r['slices']} slices ===")
    print(r["text"][:700])
    print()

---

Project: https://github.com/rekalantar/MedGemma-CT-Reader

The windowing, sampling and prompt-building functions above are also available as an
installable package in that repo, with tests.